<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

# LightGBM Algorithm for CICIDS-2017 Dataset
This notebook provides a robust replication of the `light_final.py` script, including oversampling for class balancing and SHAP analysis using `TreeExplainer`.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn import metrics
from collections import Counter
from sklearn.preprocessing import label_binarize
from imblearn.over_sampling import RandomOverSampler
import time
import matplotlib.pyplot as plt
import shap
import os

np.random.seed(0)
shap.initjs()

NameError: name 'y_test_res' is not defined

## Utility Functions
Defining the oversampling logic to handle class imbalance.

In [ ]:
def oversample(X_train, y_train):
    over = RandomOverSampler(sampling_strategy='minority')
    # Convert to numpy and oversample
    x_np = X_train.to_numpy()
    y_np = y_train.to_numpy()
    x_np, y_np = over.fit_resample(x_np, y_np)

    # Convert back to pandas
    x_over = pd.DataFrame(x_np, columns=X_train.columns)
    y_over = pd.Series(y_np)
    return x_over, y_over

## Defining Features of Interest
We use the 15 features identified as most significant in the research.

In [ ]:
req_cols = [
    ' Packet Length Std', ' Total Length of Bwd Packets', ' Subflow Bwd Bytes',
    ' Destination Port', ' Packet Length Variance', ' Bwd Packet Length Mean',
    ' Avg Bwd Segment Size', 'Bwd Packet Length Max', ' Init_Win_bytes_backward',
    'Total Length of Fwd Packets', ' Subflow Fwd Bytes', 'Init_Win_bytes_forward',
    ' Average Packet Size', ' Packet Length Mean', ' Max Packet Length', ' Label'
]

## Loading Database

In [ ]:
db_path = '../cicids_db/'
files = [
    'Wednesday-workingHours.pcap_ISCX.csv',
    'Tuesday-WorkingHours.pcap_ISCX.csv',
    'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
    'Monday-WorkingHours.pcap_ISCX.csv',
    'Friday-WorkingHours-Morning.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
    'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv'
]

frames = []
for f in files:
    path = os.path.join(db_path, f)
    print(f'Loading {f}...')
    df_tmp = pd.read_csv(path, usecols=req_cols, low_memory=False, encoding='cp1252')
    frames.append(df_tmp)

df = pd.concat(frames, ignore_index=True)
# Using 20% sample fraction as per original LightGBM script logic
df = df.sample(frac=0.2, random_state=0)
print("Finished loading all databases.")

## Preprocessing (Normalization and Label Grouping)
We clean the column names, handle label replacements, and perform max normalization.

In [ ]:
# Normalize column names (remove leading spaces)
df.columns = df.columns.str.strip()

# Filter out header rows if they appear in concat
df = df[df['Label'] != 'Label']

# Label replacement
y = df['Label'].replace({
    'DoS GoldenEye': 'Dos/Ddos', 'DoS Hulk': 'Dos/Ddos', 'DoS Slowhttptest': 'Dos/Ddos',
    'DoS slowloris': 'Dos/Ddos', 'Heartbleed': 'Dos/Ddos', 'DDoS': 'Dos/Ddos',
    'FTP-Patator': 'Brute Force', 'SSH-Patator': 'Brute Force',
    'Web Attack - Brute Force': 'Web Attack', 'Web Attack - Sql Injection': 'Web Attack', 'Web Attack - XSS': 'Web Attack',
    'Web Attack XSS': 'Web Attack', 'Web Attack Sql Injection': 'Web Attack', 'Web Attack Brute Force': 'Web Attack'
})

# Ensure numeric features
X = df.drop(columns=['Label'])
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Max Normalization
for col in X.columns:
    m = abs(X[col].max())
    if m != 0:
        X[col] = X[col] / m

df = X.assign(Label = y)

# Robustly fill NaNs: Numeric with 0, Label with BENIGN
for col in df.columns:
    if col == 'Label':
        df[col] = df[col].fillna('BENIGN').astype(str)
    else:
        df[col] = df[col].fillna(0)

# Remove duplicates
df = df.drop_duplicates()

print("Preprocessing complete.")

## Separating Training and Testing db & Balancing
We use an 80-20 split and apply oversampling to balance the minority classes.

In [ ]:
y = df['Label']
X = df.drop(columns=['Label'])

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=0)

print("Original class distribution (Train):", Counter(y_train))

# Balancing logic: iteratively oversample until all classes are equal to the majority size
def balance_dataset(X_data, y_data):
    counter = Counter(y_data)
    target_size = max(counter.values())
    
    # Simplified oversampling approach to match target size for all classes
    over = RandomOverSampler(sampling_strategy='not majority', random_state=0)
    X_res, y_res = over.fit_resample(X_data, y_data)
    return X_res, y_res

print("Balancing training set...")
X_train_res, y_train_res = balance_dataset(X_train, y_train)
print("Balanced class distribution (Train):", Counter(y_train_res))

print("Balancing testing set...")
X_test_res, y_test_res = balance_dataset(X_test, y_test)
print("Balanced class distribution (Test):", Counter(y_test_res))

y_train_num, label_map = pd.factorize(y_train_res)
y_test_num = y_test_res.map({name: i for i, name in enumerate(label_map)}).fillna(0).astype(int)

print("Label map:", label_map)

## Model Construction
Hyperparameters: Default `LGBMClassifier`.

In [ ]:
model = LGBMClassifier(random_state=0)
start = time.time()
model.fit(X_train_res, y_train_res)
print(f"ELAPSE TIME MODEL: {(time.time() - start)/60:.2f} min")

## Evaluation Metrics

In [ ]:
y_pred = model.predict(X_test_res)
y_score = model.predict_proba(X_test_res) if hasattr(model, 'predict_proba') else model.predict(X_test_res)
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import label_binarize
print('Accuracy:', accuracy_score(y_test_res, y_pred))

unique_test_classes = np.unique(y_test_res.astype(str))
if len(unique_test_classes) > 1:
    indices = [i for i, name in enumerate(label_map) if name in unique_test_classes]
    y_test_bin = label_binarize(y_test_res, classes=label_map[indices].tolist())
    y_score_filtered = y_score[:, indices]
    try:
        print('ROC-AUC:', roc_auc_score(y_test_bin, y_score_filtered, multi_class='ovr'))
    except: print('ROC-AUC calculation failed')


## SHAP Analysis
Calculating SHAP values for 500 samples using `TreeExplainer`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

start_index = 0
end_index = 100
test_data_for_shap = X_test_res.iloc[start_index:end_index]

print(f"Calculating SHAP values for {end_index} samples using TreeExplainer...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(test_data_for_shap)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, test_data_for_shap, class_names=label_map.tolist(), show=False, plot_type='bar')
plt.title("LightGBM Global Feature Importance (SHAP)")
plt.savefig('Light_Shap_Summary_global.png', bbox_inches='tight')
plt.show()